In [ ]:
!pip -q install "huggingface-hub==1.7.2" 2>&1 | tail -3


In [ ]:
import importlib.metadata
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

OUT = Path("/kaggle/working/fetch_probe")
OUT.mkdir(parents=True, exist_ok=True)
SCRIPT = OUT / "fetch_inputs.py"
SCRIPT.write_text('#!/usr/bin/env python3\n"""Fetch what a GPU session reads, every step bounded in time, reporting as it goes.\n\nSession a of the OmniVoice adaptation (2026-09-19) printed its last line 138 s\nin, as its downloads began, and nothing after that until it was cancelled:\nits download cell ran ``snapshot_download`` and a ``urlopen`` with no timeout\ninside the notebook\'s own process, so a stall held a T4 pair for the rest of\nthe session. Here every request has a timeout, each download is tried a\nbounded number of times and says when it finishes, how much it fetched and\nhow long it took, and the notebook runs this script under ``run_bounded``,\nwhich kills it at the stage\'s budget - so a stall ends the stage, not the week\'s\nGPU quota.\n\nThe plan is a JSON file::\n\n    {"snapshots": [{"name": "base", "repo": "k2-fsa/OmniVoice", "revision": "...",\n                    "repo_type": "model", "allow_patterns": null, "local_dir": "/tmp/omnivoice-base",\n                    "sha256": {"audio_tokenizer/model.safetensors": "..."},\n                    "forbid": ["test-*.parquet"]}],\n     "urls": [{"name": "sdpa config", "url": "https://...", "path": "/tmp/config.json",\n               "git_blob_prefix": "..."}]}\n\nand the report, written after every step so a killed run still leaves one,\nsays what arrived, its size and time, the attempts, and any check that failed.\n\n    python scripts/fetch_inputs.py --plan plan.json --report report.json\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport fnmatch\nimport hashlib\nimport json\nimport os\nimport sys\nimport time\nimport urllib.request\nfrom collections.abc import Callable\nfrom pathlib import Path\nfrom typing import Any\n\n#: Seconds a single request may go without an answer before it is abandoned.\nREQUEST_TIMEOUT = 60\nATTEMPTS = 3\nPAUSE = 20\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef git_blob(data: bytes) -> str:\n    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\\0" + data).hexdigest()\n\n\ndef gib(path: Path) -> float:\n    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 2**30\n\n\ndef attempt(name: str, action: Callable[[], Any], attempts: int = ATTEMPTS, pause: float = PAUSE,\n            log: Callable[[str], None] = print) -> tuple[Any, list[dict[str, Any]]]:\n    """Run ``action`` up to ``attempts`` times, reporting each try; re-raise the last failure."""\n    tries: list[dict[str, Any]] = []\n    for number in range(1, attempts + 1):\n        started = time.time()\n        try:\n            result = action()\n        except Exception as exc:  # noqa: BLE001 - recorded, then retried or re-raised\n            tries.append({"attempt": number, "seconds": round(time.time() - started, 1),\n                          "error": f"{type(exc).__name__}: {exc}"[:500]})\n            log(f"{name}: attempt {number} failed after {tries[-1][\'seconds\']} s - {tries[-1][\'error\']}")\n            if number == attempts:\n                raise\n            time.sleep(pause)\n            continue\n        tries.append({"attempt": number, "seconds": round(time.time() - started, 1)})\n        return result, tries\n    raise AssertionError("unreachable")\n\n\ndef fetch_url(url: str, timeout: float = REQUEST_TIMEOUT) -> bytes:\n    with urllib.request.urlopen(url, timeout=timeout) as response:\n        return response.read()\n\n\ndef run(plan: dict[str, Any], report_path: Path, log: Callable[[str], None] = print) -> dict[str, Any]:\n    from huggingface_hub import snapshot_download\n\n    report: dict[str, Any] = {"started": time.strftime("%Y-%m-%d %H:%M:%S %z"), "steps": [], "ok": False}\n\n    def save() -> None:\n        report_path.write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n\n    save()\n    for item in plan.get("urls", []):\n        step = {"name": item["name"], "kind": "url"}\n        report["steps"].append(step)\n        data, step["tries"] = attempt(item["name"], lambda item=item: fetch_url(item["url"]), log=log)\n        blob = git_blob(data)\n        if item.get("git_blob_prefix") and not blob.startswith(item["git_blob_prefix"]):\n            step["failed_check"] = f"git blob {blob} does not start with {item[\'git_blob_prefix\']}"\n            save()\n            raise SystemExit(f"{item[\'name\']}: {step[\'failed_check\']}")\n        Path(item["path"]).parent.mkdir(parents=True, exist_ok=True)\n        Path(item["path"]).write_bytes(data)\n        step.update({"bytes": len(data), "git_blob": blob})\n        log(f"{item[\'name\']}: {len(data)} bytes, git blob {blob[:12]}")\n        save()\n    for item in plan.get("snapshots", []):\n        step = {"name": item["name"], "kind": "snapshot", "repo": item["repo"], "revision": item["revision"]}\n        report["steps"].append(step)\n        started = time.time()\n        folder, step["tries"] = attempt(item["name"], lambda item=item: snapshot_download(\n            item["repo"], repo_type=item.get("repo_type", "model"), revision=item["revision"],\n            allow_patterns=item.get("allow_patterns"), local_dir=item["local_dir"]), log=log)\n        folder = Path(folder)\n        size = gib(folder)\n        step.update({"bytes": sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()),\n                     "gib": round(size, 3), "seconds": round(time.time() - started, 1),\n                     "mib_per_second": round(size * 1024 / max(time.time() - started, 1e-6), 1)})\n        forbidden = [str(p.relative_to(folder)) for pattern in item.get("forbid", [])\n                     for p in folder.rglob("*") if fnmatch.fnmatch(p.name, pattern)]\n        if forbidden:\n            step["failed_check"] = f"forbidden files arrived: {forbidden[:5]}"\n            save()\n            raise SystemExit(f"{item[\'name\']}: {step[\'failed_check\']}")\n        for relative, expected in (item.get("sha256") or {}).items():\n            actual = sha256(folder / relative)\n            if actual != expected:\n                step["failed_check"] = f"{relative} has sha256 {actual}, expected {expected}"\n                save()\n                raise SystemExit(f"{item[\'name\']}: {step[\'failed_check\']}")\n        log(f"{item[\'name\']}: {size:.2f} GiB in {step[\'seconds\']:.0f} s ({step[\'mib_per_second\']} MiB/s)")\n        save()\n    report["ok"] = True\n    report["finished"] = time.strftime("%Y-%m-%d %H:%M:%S %z")\n    save()\n    return report\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    parser.add_argument("--plan", type=Path, required=True)\n    parser.add_argument("--report", type=Path, required=True)\n    parser.add_argument("--no-xet", action="store_true",\n                        help="Download over plain HTTP rather than Hugging Face\'s Xet client")\n    args = parser.parse_args(argv)\n    # Every Hugging Face request gives up after REQUEST_TIMEOUT seconds instead of waiting for good.\n    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", str(REQUEST_TIMEOUT))\n    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", str(REQUEST_TIMEOUT))\n    if args.no_xet:\n        os.environ["HF_HUB_DISABLE_XET"] = "1"\n\n    def log(line: str) -> None:\n        print(f"[{time.strftime(\'%H:%M:%S\')}] {line}", flush=True)\n\n    report = run(json.loads(args.plan.read_text(encoding="utf-8")), args.report, log=log)\n    log("all fetched" if report["ok"] else "incomplete")\n    return 0 if report["ok"] else 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
PLAN = json.loads("{\"urls\": [{\"name\": \"train_config_finetune_sdpa.json\", \"git_blob_prefix\": \"4b0a543e23be\", \"path\": \"/tmp/adapt/official/train_config_finetune_sdpa.json\", \"url\": \"https://raw.githubusercontent.com/k2-fsa/OmniVoice/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/examples/config/train_config_finetune_sdpa.json\"}], \"snapshots\": [{\"name\": \"base\", \"repo\": \"k2-fsa/OmniVoice\", \"revision\": \"c5fdb5ccb189668d56333f77ba2629f4cd7535f4\", \"local_dir\": \"/tmp/omnivoice-base\", \"sha256\": {\"audio_tokenizer/model.safetensors\": \"fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d\"}}, {\"name\": \"audio tokenizer\", \"repo\": \"eustlb/higgs-audio-v2-tokenizer\", \"revision\": \"528e871c2a26c4f0f7773b9754e2e1acae20899d\", \"local_dir\": \"/tmp/higgs-audio-v2-tokenizer\", \"sha256\": {\"model.safetensors\": \"fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d\"}}, {\"name\": \"common voice 26\", \"repo\": \"pymmdrza/Common-Voice-Speech-26.0-Persian-Clean\", \"repo_type\": \"dataset\", \"revision\": \"142e9bb40b56d9491faa0ec0987ee8d408bd9c05\", \"allow_patterns\": [\"data/*\"], \"local_dir\": \"/tmp/cv26\"}, {\"name\": \"neyshekar train and valid\", \"repo\": \"shekar-ai/neyshekar\", \"repo_type\": \"dataset\", \"revision\": \"30696ada746a66bb3047c984744b5ecfd60fe4e9\", \"allow_patterns\": [\"data/train-*\", \"data/validation-*\"], \"local_dir\": \"/tmp/neyshekar\", \"forbid\": [\"test-*.parquet\"]}]}")
FOLDERS = sorted({step["local_dir"] for step in PLAN["snapshots"]} | {"/tmp/adapt"})


def versions():
    found = {}
    for name in ("huggingface-hub", "hf-xet", "requests", "httpx"):
        try:
            found[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            found[name] = None
    return found


def clear():
    for folder in FOLDERS:
        shutil.rmtree(folder, ignore_errors=True)
    # Only on Kaggle: anywhere else this cache holds models someone downloaded on purpose.
    if Path("/kaggle/working").is_dir():
        shutil.rmtree(Path.home() / ".cache/huggingface", ignore_errors=True)


RESULT = {"versions": versions(), "disk_free_gib": round(shutil.disk_usage("/tmp").free / 2**30, 1),
           "plan": [step["name"] for step in PLAN["snapshots"]], "runs": {}}
print(json.dumps(RESULT, indent=1), flush=True)
(OUT / "plan.json").write_text(json.dumps(PLAN, indent=1))
for mode, flags in (("xet", []), ("http", ["--no-xet"])):
    clear()
    started = time.time()
    log = OUT / f"{mode}.log"
    report = OUT / f"{mode}.json"
    with log.open("wb") as sink:
        process = subprocess.Popen([sys.executable, str(SCRIPT), "--plan", str(OUT / "plan.json"),
                                    "--report", str(report), *flags], stdout=sink, stderr=subprocess.STDOUT)
        try:
            code = process.wait(timeout=30 * 60)
            timed_out = False
        except subprocess.TimeoutExpired:
            process.kill()
            code, timed_out = process.wait(), True
    finished = json.loads(report.read_text()) if report.is_file() else {}
    RESULT["runs"][mode] = {
        "exit": code, "timed_out": timed_out, "minutes": round((time.time() - started) / 60, 2),
        "ok": finished.get("ok"),
        "steps": {s["name"]: {k: s.get(k) for k in ("gib", "seconds", "mib_per_second", "tries", "failed_check")}
                  for s in finished.get("steps", [])},
    }
    print(mode, json.dumps(RESULT["runs"][mode], indent=1), flush=True)
clear()
(OUT / "result.json").write_text(json.dumps(RESULT, indent=2))
print(json.dumps(RESULT, indent=2))
